# Perspective + OCR Dual-CNN

End-to-end model that reads a credit card number from a photo taken at an angle, without
being told where the card is.

1. **PerspectiveNet** — predicts the card's four corners
2. **Spatial Transformer** — builds the homography from those corners and applies it with
   `grid_sample` (differentiable, so the OCR loss reaches the geometry)
3. **OCRNet** — reads 16 digits from the straightened crop

## Why corners and not the matrix

The obvious approach — regress the 9 entries of the homography and take an MSE — does not
work, and it fails in a way that looks like it is working. Homography entries are on wildly
different scales and act non-linearly: the projective terms sit around `1e-5` and appear in
the *denominator* of the projection. A regression that explains 94% of their variance can
still place the sampled crop thousands of pixels off the card.

Measured on this dataset, using the ground-truth matrices:

| parameterisation | error | resulting crop shift |
|---|---|---|
| 9 matrix entries | 0.24σ per entry (MSE 0.058) | **~17,000 px** — off the image entirely |
| 4 corners | 1% of card width (14 px) | 15 px — a third of one digit |
| 4 corners | 5% of card width (69 px) | 70 px — 1.5 digits |

Corner error maps to crop error roughly linearly and degrades gracefully; matrix-entry error
does not degrade, it explodes. So the network predicts **8 numbers — four corner positions in
normalised image coordinates** — and the homography is recovered from them by a differentiable
DLT solve. This is the 4-point parameterisation from DeTone et al., *Deep Image Homography
Estimation*.

**Loss** = `alpha · MSE(corners)` + `beta · CrossEntropy(digits)`, with the corner term now in
directly interpretable units.

In [ ]:
import os
import json

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [ ]:
RECT_W, RECT_H = 512, 323   # the rectangle the ground-truth matrices map the card into
IMG_SIZE = (256, 256)       # what the network sees
CARD_SIZE = (64, 128)       # (H, W) of the straightened crop fed to the OCR head

# destination corners, in the order top-left, top-right, bottom-right, bottom-left
DST_CORNERS = torch.tensor([[0.0, 0.0],
                            [RECT_W - 1, 0.0],
                            [RECT_W - 1, RECT_H - 1],
                            [0.0, RECT_H - 1]])

## Dataset

The stored ground truth is the 3×3 matrix, but the corners are what we actually supervise.
Since the matrix maps the card quad onto the fixed rectangle, applying its inverse to the
rectangle's corners recovers the quad exactly (verified to ~5e-6 px). So no extra files are
needed — and the corners are normalised to `[-1, 1]` here, which is the coordinate space
`grid_sample` uses, so the model never has to know the original image resolution.

In [ ]:
class PerspectiveOCRDataset(Dataset):
    """
    Each sample provides:
      - image   : (3, 256, 256), normalised to [-1, 1]
      - corners : (4, 2) card corners in normalised [-1, 1] image coordinates
      - label   : (16,) digit indices
    """

    def __init__(self, img_dir, matrix_dir, label_dir, img_size=IMG_SIZE, orig_sizes=None):
        self.img_dir, self.matrix_dir, self.label_dir = img_dir, matrix_dir, label_dir
        self.image_files = sorted(f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png')))
        if not self.image_files:
            raise FileNotFoundError(f"No images found in {img_dir}")

        # The matrices are in ORIGINAL-resolution pixel coordinates. If the images have been
        # pre-resized for speed their own size no longer reveals that, so it is read from
        # orig_sizes.json when available.
        self.orig_sizes = None
        if orig_sizes and os.path.exists(orig_sizes):
            with open(orig_sizes) as f:
                self.orig_sizes = json.load(f)
            print(f"Using original sizes from {orig_sizes}")

        self.chars = "0123456789"
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.idx_to_char = {i: c for i, c in enumerate(self.chars)}

        rect = DST_CORNERS.numpy().astype(np.float64)
        self._rect_h = np.hstack([rect, np.ones((4, 1))])

        self.transform = transforms.Compose([
            transforms.Resize(img_size),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])

        self._filter_degenerate()

    def _filter_degenerate(self, min_area=0.05):
        """
        Drop samples whose card is unusable.

        Stage 2 of the generation pipeline applies random affines and a few slip through
        collapsed to a sliver, pushed outside the frame, or geometrically degenerate. A
        131x17 px card cannot be read by anything, and a degenerate quad makes the
        homography undefined, so these are removed rather than trained on.
        """
        keep, reasons = [], {'tiny': 0, 'outside': 0, 'degenerate': 0}
        for fname in self.image_files:
            stem = os.path.splitext(fname)[0]
            corners, raw = self._corners_for(stem, with_raw=True)
            c = corners.numpy()

            area = 0.5 * abs(float(np.dot(c[:, 0], np.roll(c[:, 1], -1))
                                   - np.dot(c[:, 1], np.roll(c[:, 0], -1))))
            dx1, dx2 = c[1, 0] - c[2, 0], c[3, 0] - c[2, 0]
            dy1, dy2 = c[1, 1] - c[2, 1], c[3, 1] - c[2, 1]
            den = abs(dx1 * dy2 - dy1 * dx2)

            if area < min_area:
                reasons['tiny'] += 1
            elif np.abs(c).max() > 1.0:
                reasons['outside'] += 1
            elif den < 1e-6:
                reasons['degenerate'] += 1
            else:
                keep.append(fname)

        dropped = len(self.image_files) - len(keep)
        if dropped:
            print(f"Dropped {dropped} unusable samples "
                  f"({reasons['tiny']} tiny, {reasons['outside']} outside frame, "
                  f"{reasons['degenerate']} degenerate)")
        self.image_files = keep

    def encode_label(self, text):
        clean = text.replace(' ', '').strip()
        return torch.tensor([self.char_to_idx[c] for c in clean if c in self.char_to_idx],
                            dtype=torch.long)

    def corners_from_matrix(self, matrix, width, height):
        """Invert the matrix onto the rectangle's corners, then normalise to [-1, 1]."""
        pts = np.linalg.solve(matrix.astype(np.float64), self._rect_h.T).T
        pts = pts[:, :2] / pts[:, 2:3]
        pts[:, 0] = 2.0 * pts[:, 0] / (width - 1) - 1.0
        pts[:, 1] = 2.0 * pts[:, 1] / (height - 1) - 1.0
        return torch.from_numpy(pts.astype(np.float32))

    def _corners_for(self, stem, with_raw=False):
        matrix = np.load(os.path.join(self.matrix_dir, f"{stem}.npy"))
        if self.orig_sizes is not None:
            width, height = self.orig_sizes[stem]
        else:
            with Image.open(os.path.join(self.img_dir, stem + '.jpg')) as im:
                width, height = im.size
        corners = self.corners_from_matrix(matrix, width, height)
        return (corners, matrix) if with_raw else corners

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        stem = os.path.splitext(img_name)[0]

        image = self.transform(Image.open(os.path.join(self.img_dir, img_name)).convert('RGB'))

        corners = self._corners_for(stem)

        with open(os.path.join(self.label_dir, f"{stem}.txt")) as f:
            label = self.encode_label(f.read().strip())

        return {'image': image, 'corners': corners, 'label': label}


def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    images = torch.stack([b['image'] for b in batch], 0)
    corners = torch.stack([b['corners'] for b in batch], 0)
    labels = pad_sequence([b['label'] for b in batch], batch_first=True, padding_value=0)
    return images, corners, labels

## Load dataset

In [ ]:
USE_COLAB = True

# Google Drive file id of the dataset zip (the part of the share link after /d/).
DATA_ZIP_ID = "14l6hsSd_iGI3zAd8_0o9bvMSscATDVel"

if USE_COLAB:
    if not DATA_ZIP_ID:
        raise ValueError("Set DATA_ZIP_ID to the Drive file id of your dataset zip.")
    !pip install -q gdown
    !gdown -q "https://drive.google.com/uc?id={DATA_ZIP_ID}" -O data.zip
    !unzip -q -o data.zip -d /content/data
    SEARCH_ROOT = "/content/data"
else:
    SEARCH_ROOT = "."


def find_dataset_root(search_root):
    """Find the extracted dataset without caring how the zip nested its contents."""
    needed = {"images", "matrices", "labels"}
    for current, dirs, _ in os.walk(search_root):
        if needed.issubset(set(dirs)):
            return current
    raise FileNotFoundError(
        f"No folder under {search_root} contains images/, matrices/ and labels/.\n"
        f"Check what was extracted with:  !find {search_root} -maxdepth 3 -type d")


ROOT = find_dataset_root(SEARCH_ROOT)
print(f"Dataset root: {ROOT}")

dataset = PerspectiveOCRDataset(f"{ROOT}/images", f"{ROOT}/matrices", f"{ROOT}/labels",
                                img_size=IMG_SIZE, orig_sizes=f"{ROOT}/orig_sizes.json")

sample = dataset[0]
print(f"Dataset size : {len(dataset)}")
print(f"Image        : {tuple(sample['image'].shape)}")
print(f"Label        : {sample['label'].tolist()}")
print(f"Corners      :\n{sample['corners']}")

# Mean quad across the dataset — used to initialise the corner regressor.
mean_corners = torch.stack([dataset[i]['corners'] for i in range(len(dataset))]).mean(0)
print(f"\nDataset mean corners:\n{mean_corners}")

## Model

In [ ]:
def homography_unit_to_quad(corners, eps=1e-8):
    """
    Homography mapping the unit square onto `corners`, in closed form (Heckbert).

    Solving an 8x8 DLT system for this is both slower and numerically fragile: with the
    destination in pixel units the system's condition number on this dataset has a median of
    2,200 and a maximum of 4e13, which is singular in float32 — the first training attempt
    died on exactly that. The closed form has a single division and reproduces the corners to
    machine precision on every non-degenerate sample.

    Args:
        corners: (B, 4, 2) quad in normalised [-1, 1] coordinates, ordered TL, TR, BR, BL
    Returns:
        (B, 3, 3) homography taking the unit square to that quad
    """
    x0, y0 = corners[:, 0, 0], corners[:, 0, 1]
    x1, y1 = corners[:, 1, 0], corners[:, 1, 1]
    x2, y2 = corners[:, 2, 0], corners[:, 2, 1]
    x3, y3 = corners[:, 3, 0], corners[:, 3, 1]

    dx1, dx2, dx3 = x1 - x2, x3 - x2, x0 - x1 + x2 - x3
    dy1, dy2, dy3 = y1 - y2, y3 - y2, y0 - y1 + y2 - y3

    den = dx1 * dy2 - dy1 * dx2
    den = torch.where(den.abs() < eps, torch.full_like(den, eps), den)

    g = (dx3 * dy2 - dy3 * dx2) / den
    h = (dx1 * dy3 - dy1 * dx3) / den

    return torch.stack([
        torch.stack([x1 - x0 + g * x1, x3 - x0 + h * x3, x0], dim=-1),
        torch.stack([y1 - y0 + g * y1, y3 - y0 + h * y3, y0], dim=-1),
        torch.stack([g, h, torch.ones_like(g)], dim=-1),
    ], dim=1)

In [ ]:
class PerspectiveNet(nn.Module):
    """Predicts the card's four corners in normalised [-1, 1] image coordinates."""

    def __init__(self, in_channels=3, init_corners=None):
        super().__init__()

        def block(cin, cout, pool=True):
            layers = [nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout),
                      nn.ReLU(inplace=True)]
            if pool:
                layers.append(nn.MaxPool2d(2, 2))
            return layers

        self.features = nn.Sequential(
            *block(in_channels, 32),   # 256 -> 128
            *block(32, 64),            # 128 -> 64
            *block(64, 128),           # 64  -> 32
            *block(128, 256),          # 32  -> 16
            *block(256, 512),          # 16  -> 8
            *block(512, 512, pool=False),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.regressor = nn.Sequential(
            nn.Linear(512 * 4 * 4, 1024), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(1024, 256), nn.ReLU(inplace=True),
            nn.Linear(256, 8),
        )

        # Start every prediction at the dataset-mean quad rather than a random one, so the
        # very first crops already overlap the card instead of sampling empty background.
        nn.init.zeros_(self.regressor[-1].weight)
        if init_corners is None:
            init_corners = torch.tensor([[-0.5, -0.25], [0.5, -0.25], [0.5, 0.25], [-0.5, 0.25]])
        self.regressor[-1].bias.data = init_corners.reshape(-1).clone().float()

    def forward(self, x):
        x = self.features(x)
        return self.regressor(x.flatten(1)).view(-1, 4, 2)

In [ ]:
class SpatialTransformer(nn.Module):
    """
    Straightens the card by sampling the unit square through the corner homography.

    Because the homography maps the unit square *onto* the card, the destination grid is the
    unit square and the mapping runs forward — there is no matrix inverse and no linear solve
    anywhere in the training loop. Everything is in normalised [-1, 1] image coordinates,
    which is the space grid_sample works in, so the original image resolution never enters.
    """

    def __init__(self, output_size=CARD_SIZE):
        super().__init__()
        self.out_h, self.out_w = output_size
        vs, us = torch.meshgrid(torch.linspace(0, 1, self.out_h),
                                torch.linspace(0, 1, self.out_w), indexing='ij')
        grid = torch.stack([us.flatten(), vs.flatten(), torch.ones(us.numel())])
        self.register_buffer('unit_grid', grid)          # (3, N)

    def forward(self, image, corners):
        B = image.size(0)
        H = homography_unit_to_quad(corners)
        grid = self.unit_grid.unsqueeze(0).expand(B, -1, -1).to(image)

        src = torch.bmm(H, grid)                          # unit square -> normalised source
        denom = src[:, 2, :]
        denom = torch.where(denom.abs() < 1e-8, torch.full_like(denom, 1e-8), denom)
        norm = torch.stack([src[:, 0, :] / denom, src[:, 1, :] / denom], dim=-1)

        return F.grid_sample(image, norm.view(B, self.out_h, self.out_w, 2),
                             mode='bilinear', padding_mode='zeros', align_corners=True)

In [ ]:
class OCRNet(nn.Module):
    """
    Reads 16 digits from a (B, 3, 64, 128) straightened crop.

    The last two pools are height-only. Pooling width five times would leave a 4-wide feature
    map, and adaptively pooling that up to 16 positions would hand four consecutive digits
    identical features — a hard accuracy ceiling no amount of training can lift.
    """

    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                                    # 64x128 -> 32x64
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                                    # -> 16x32
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                                    # -> 8x16
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1)),                                  # -> 4x16  (width kept)
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1)),                                  # -> 2x16  (width kept)
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 16)),                         # -> 1x16
        )
        self.classifier = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Linear(256, 128), nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x).squeeze(2)               # (B, 512, 16)
        return self.classifier(x.permute(0, 2, 1))    # (B, 16, 10)

### Teacher forcing on the geometry

There is a chicken-and-egg problem here. The OCR head can only learn from crops that actually
contain the card, but early on the predicted corners are wrong, so it sees background. It
learns nothing, produces no useful gradient, and the geometry gets no help from it either.

So for the first epochs the OCR head is fed the crop from the **ground-truth** corners, and
the probability of that is annealed to zero. By the time the model is on its own predictions,
the OCR head can already read a well-formed crop and its gradients are meaningful.

In [ ]:
class PerspectiveOCRModel(nn.Module):
    def __init__(self, card_size=CARD_SIZE, init_corners=None):
        super().__init__()
        self.perspective_net = PerspectiveNet(in_channels=3, init_corners=init_corners)
        self.spatial_transformer = SpatialTransformer(output_size=card_size)
        self.ocr_net = OCRNet(in_channels=3, num_classes=10)

    def forward(self, x, gt_corners=None, teacher_prob=0.0):
        pred_corners = self.perspective_net(x)
        straightened = self.spatial_transformer(x, pred_corners)

        ocr_input = straightened
        if self.training and gt_corners is not None and teacher_prob > 0:
            gt_straightened = self.spatial_transformer(x, gt_corners)
            use_gt = (torch.rand(x.size(0), device=x.device) < teacher_prob)
            mask = use_gt.view(-1, 1, 1, 1).to(straightened.dtype)
            ocr_input = mask * gt_straightened + (1 - mask) * straightened

        return pred_corners, straightened, self.ocr_net(ocr_input)

## Sanity check before training

Warp with the **ground-truth** corners. If these are not readable cards, nothing downstream
can work, and it is far cheaper to discover that here than after 50 epochs.

In [ ]:
def show_gt_straightening(dataset, n=4):
    st = SpatialTransformer()
    fig, axes = plt.subplots(n, 2, figsize=(10, 2.4 * n))
    denorm = lambda t: (t * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()

    for row in range(n):
        s = dataset[row * (len(dataset) // n)]
        with torch.no_grad():
            out = st(s['image'].unsqueeze(0), s['corners'].unsqueeze(0))
        digits = ''.join(str(d) for d in s['label'].tolist())
        axes[row, 0].imshow(denorm(s['image'])); axes[row, 0].set_title('input'); axes[row, 0].axis('off')
        axes[row, 1].imshow(denorm(out[0]))
        axes[row, 1].set_title(' '.join(digits[i:i + 4] for i in range(0, 16, 4)))
        axes[row, 1].axis('off')

    plt.tight_layout(); plt.show()


show_gt_straightening(dataset)

## Training

In [ ]:
def corner_error_px(pred, gt, image_width=RECT_W):
    """Mean corner distance, expressed in pixels of the straightened card for readability."""
    return ((pred - gt).pow(2).sum(-1).sqrt().mean() * image_width / 2).item()


def run_epoch(model, loader, alpha, beta, device, teacher_prob=0.0, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    mse_criterion, ce_criterion = nn.MSELoss(), nn.CrossEntropyLoss()
    totals = np.zeros(4)

    with torch.set_grad_enabled(train):
        for images, gt_corners, gt_labels in loader:
            images = images.to(device)
            gt_corners = gt_corners.to(device)
            gt_labels = gt_labels.to(device)

            pred_corners, _, digit_logits = model(
                images, gt_corners=gt_corners, teacher_prob=teacher_prob if train else 0.0)

            loss_corner = mse_criterion(pred_corners, gt_corners)
            nd = min(digit_logits.size(1), gt_labels.size(1))
            loss_ce = ce_criterion(digit_logits[:, :nd, :].reshape(-1, 10),
                                   gt_labels[:, :nd].reshape(-1))
            loss = alpha * loss_corner + beta * loss_ce

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()

            totals += np.array([loss.item(), loss_corner.item(), loss_ce.item(),
                                corner_error_px(pred_corners, gt_corners)])

    return totals / len(loader)


def train_model(model, train_loader, val_loader, num_epochs=50, lr=1e-4, device='cuda',
                alpha=10.0, beta=1.0, teacher_epochs=15):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    history = {k: [] for k in ('train_loss', 'val_loss', 'train_corner', 'val_corner',
                               'train_ce', 'val_ce', 'val_corner_px')}

    print(f"Training on {device} | alpha={alpha}, beta={beta}, teacher_epochs={teacher_epochs}")
    print(f"{'Epoch':>6} {'Teacher':>8} {'Train':>9} {'Val':>9} {'Val CE':>8} {'Corner px':>10}")
    print("-" * 56)

    for epoch in range(num_epochs):
        teacher_prob = max(0.0, 1.0 - epoch / teacher_epochs) if teacher_epochs else 0.0
        tr = run_epoch(model, train_loader, alpha, beta, device, teacher_prob, optimizer)
        va = run_epoch(model, val_loader, alpha, beta, device)
        scheduler.step(va[0])

        for k, v in zip(('train_loss', 'train_corner', 'train_ce'), tr[:3]): history[k].append(v)
        for k, v in zip(('val_loss', 'val_corner', 'val_ce'), va[:3]): history[k].append(v)
        history['val_corner_px'].append(va[3])

        if epoch == 0 or (epoch + 1) % 5 == 0:
            print(f"{epoch+1:>6} {teacher_prob:>8.2f} {tr[0]:>9.4f} {va[0]:>9.4f} "
                  f"{va[2]:>8.4f} {va[3]:>10.1f}")

    return model, history

In [ ]:
BATCH_SIZE, LEARNING_RATE, EPOCHS = 16, 1e-4, 50
ALPHA, BETA, TEACHER_EPOCHS = 10.0, 1.0, 15
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(42)
generator = torch.Generator().manual_seed(42)
train_size = int(0.8 * len(dataset))
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, len(dataset) - train_size], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn, num_workers=2, pin_memory=True)

model = PerspectiveOCRModel(init_corners=mean_corners)
print(f"Train: {train_size}, Val: {len(dataset) - train_size}, Device: {DEVICE}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}\n")

model, history = train_model(model, train_loader, val_loader, num_epochs=EPOCHS,
                             lr=LEARNING_RATE, device=DEVICE, alpha=ALPHA, beta=BETA,
                             teacher_epochs=TEACHER_EPOCHS)

In [ ]:
def plot_history(history):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (key, title) in zip(axes, [('loss', 'Total Loss'),
                                       ('corner', 'Corner MSE'),
                                       ('ce', 'OCR CrossEntropy')]):
        ax.plot(history[f'train_{key}'], label='Train')
        ax.plot(history[f'val_{key}'], label='Val')
        ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True)
    plt.tight_layout(); plt.show()

    plt.figure(figsize=(6, 4))
    plt.plot(history['val_corner_px'])
    plt.title('Validation corner error (px of straightened card)')
    plt.xlabel('Epoch'); plt.grid(True); plt.show()


plot_history(history)

## Evaluation

In [ ]:
def evaluate_accuracy(model, loader, device, num_digits=16):
    model.eval().to(device)
    correct_seq = correct_dig = total_dig = total = 0
    corner_px = 0.0

    with torch.no_grad():
        for images, gt_corners, gt_labels in loader:
            images, gt_corners = images.to(device), gt_corners.to(device)
            gt_labels = gt_labels.to(device)

            pred_corners, _, digit_logits = model(images)
            corner_px += corner_error_px(pred_corners, gt_corners)

            nd = min(digit_logits.size(1), gt_labels.size(1), num_digits)
            matches = digit_logits[:, :nd, :].argmax(dim=2).eq(gt_labels[:, :nd])
            correct_seq += matches.all(dim=1).sum().item()
            correct_dig += matches.sum().item()
            total_dig += matches.numel()
            total += images.size(0)

    seq_acc, dig_acc = 100.0 * correct_seq / total, 100.0 * correct_dig / total_dig
    print("=" * 46)
    print(f"Samples             : {total}")
    print(f"Sequence accuracy   : {seq_acc:.2f}%  ({correct_seq}/{total})")
    print(f"Per-digit accuracy  : {dig_acc:.2f}%")
    print(f"Mean corner error   : {corner_px / len(loader):.1f} px of the straightened card")
    print("=" * 46)
    return seq_acc, dig_acc, corner_px / len(loader)


evaluate_accuracy(model, val_loader, DEVICE)

## Predictions

Input, the crop from the ground-truth corners, and the crop the model produced. If the middle
column is readable and the right one is not, geometry is the bottleneck; if both are readable
and the digits are still wrong, the OCR head is.

In [ ]:
def visualize_predictions(model, dataset, device, num_samples=6, seed=0):
    model.eval().to(device)
    st = SpatialTransformer().to(device)
    indices = np.random.default_rng(seed).choice(len(dataset), num_samples, replace=False)

    fig, axes = plt.subplots(num_samples, 3, figsize=(13, 2.6 * num_samples))
    denorm = lambda t: (t * 0.5 + 0.5).clamp(0, 1).cpu().permute(1, 2, 0).numpy()
    group = lambda s: ' '.join(s[i:i + 4] for i in range(0, len(s), 4))

    for row, idx in enumerate(indices):
        s = dataset[int(idx)]
        image = s['image'].unsqueeze(0).to(device)

        with torch.no_grad():
            _, pred_crop, digit_logits = model(image)
            gt_crop = st(image, s['corners'].unsqueeze(0).to(device))

        pred_text = group(''.join(str(d) for d in digit_logits.argmax(dim=2)[0].cpu().tolist()))
        gt_text = group(''.join(str(d) for d in s['label'].tolist()))
        mark = '✓' if pred_text == gt_text else '✗'

        for col, (img, title) in enumerate([
                (denorm(s['image']), 'input'),
                (denorm(gt_crop[0]), f'GT corners\n{gt_text}'),
                (denorm(pred_crop[0]), f'predicted {mark}\n{pred_text}')]):
            axes[row, col].imshow(img); axes[row, col].set_title(title, fontsize=9)
            axes[row, col].axis('off')

    plt.tight_layout(); plt.show()


visualize_predictions(model, dataset, DEVICE)

## Save

In [ ]:
torch.save(model.state_dict(), 'perspective_ocr_model.pth')
print('Saved to perspective_ocr_model.pth')